# Notebook 02 — Feature Engineering and Target Construction for CLD Modeling

## Goal

This notebook builds the canonical clone-level dataset used for downstream machine learning.

It performs two linked tasks:

### 1. Feature engineering
Transform raw early-stage assay measurements into clone-level model inputs.

- **Feature engineering** means converting raw measurements into variables that a model can use.
- Examples:
  - mean
  - standard deviation
  - slope
  - last observed value
  - qP proxy

### 2. Target construction
Compute late-stage clone outcomes from the raw database.

- **Target construction** means defining the values the model should predict.
- In this project, the main targets are:
  - late qP
  - qP drop
  - stability label
  - late aggregation

---

## Important principle: leakage-safe design

This notebook uses only **early-stage information** to build features.

- **Leakage** (or **target leakage**) means future information accidentally enters the feature set.
- That would make model performance look unrealistically good.

So the workflow is:

- early passages → features
- late passages → targets only

---

## Important term: qP proxy

In this notebook, productivity is represented using a **qP proxy**:

- `qP_proxy = titer / VCD`

This should be interpreted as a **relative productivity measure**
for clone comparison, not as a fully calibrated physical qP unit.

---

## Output

This notebook saves:

1. **features-only table**
2. **canonical modeling dataset**

The final dataset is used by:

- Notebook 03b — prediction engine
- Notebook 04b — decision engine

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from pathlib import Path

scenario = "legacy"   # or "optimized"
n_clones = 5000

EARLY_START = 3
EARLY_END = 10
LATE_START = 24
LATE_END = 30
STABILITY_DROP_THRESHOLD = 0.30

DB_PATH = Path(f"../data/synthetic/raw/cld_{n_clones}clones_{scenario}.db")
OUT_DIR = Path("../data/synthetic/processed")

OUT_FEATURES = OUT_DIR / f"cld_features_v2_{n_clones}_{scenario}.csv"
OUT_DATASET = OUT_DIR / (
    f"cld_features_with_labels_qp_targets_v2_{LATE_START}_{LATE_END}_{n_clones}_{scenario}.csv"
)

print("DB:", DB_PATH)
print("Output features:", OUT_FEATURES)
print("Output dataset:", OUT_DATASET)
print("Early window:", EARLY_START, EARLY_END)
print("Late window:", LATE_START, LATE_END)
print("Stable threshold:", STABILITY_DROP_THRESHOLD)

DB: ../data/synthetic/raw/cld_5000clones_legacy.db
Output features: ../data/synthetic/processed/cld_features_v2_5000_legacy.csv
Output dataset: ../data/synthetic/processed/cld_features_with_labels_qp_targets_v2_24_30_5000_legacy.csv
Early window: 3 10
Late window: 24 30
Stable threshold: 0.3


## 1. Load raw assay data

We load assay-level data directly from the SQLite database.

This raw data will be used to build:

- early-stage features
- late-stage targets

Only early passages will be used for feature engineering.
Late passages will only be used for target construction.

In [2]:
conn = sqlite3.connect(DB_PATH)

assay = pd.read_sql_query("""
SELECT
    ar.assay_id,
    ar.assay_type,
    ar.value,
    ar.unit,
    ar.method,
    ar.batch_id,
    p.clone_id,
    p.passage_number,
    p.phase
FROM assay_result ar
JOIN passage p
  ON p.passage_id = ar.passage_id
""", conn)

print("Assay rows:", len(assay))
display(assay.head())

Assay rows: 605000


,assay_id,assay_type,value,unit,method,batch_id,clone_id,passage_number,phase
0,ASSAY_CLONE_0001_P01_titer,titer,3.728871e+00,g/L,ELISA,B_P01,CLONE_0001,1,early
1,ASSAY_CLONE_0001_P01_vcd,vcd,9.886511e+06,cells/mL,Vi-CELL,B_P01,CLONE_0001,1,early
2,ASSAY_CLONE_0001_P01_viability,viability,9.360340e+01,%,Vi-CELL,B_P01,CLONE_0001,1,early
3,ASSAY_CLONE_0001_P01_aggregation,aggregation,5.650827e+00,%,SEC-HPLC,B_P01,CLONE_0001,1,early
4,ASSAY_CLONE_0001_P02_titer,titer,3.558928e+00,g/L,ELISA,B_P02,CLONE_0001,2,early


## 2. Restrict to early window

We use only passages P3–10 for feature engineering.

This mimics real CLD decision timing:
- early passages are available at selection time
- late passages are not

In [3]:
assay_early = assay[
    (assay["passage_number"] >= EARLY_START) &
    (assay["passage_number"] <= EARLY_END)
].copy()

print("Early assay rows:", len(assay_early))
display(assay_early.head())

Early assay rows: 165000


,assay_id,assay_type,value,unit,method,batch_id,clone_id,passage_number,phase
8,ASSAY_CLONE_0001_P03_titer,titer,3.677143e+00,g/L,ELISA,B_P03,CLONE_0001,3,early
9,ASSAY_CLONE_0001_P03_vcd,vcd,1.134129e+07,cells/mL,Vi-CELL,B_P03,CLONE_0001,3,early
10,ASSAY_CLONE_0001_P03_viability,viability,9.069588e+01,%,Vi-CELL,B_P03,CLONE_0001,3,early
11,ASSAY_CLONE_0001_P03_aggregation,aggregation,5.468207e+00,%,SEC-HPLC,B_P03,CLONE_0001,3,early
12,ASSAY_CLONE_0001_P03_ddpcr_cn,ddpcr_cn,4.000000e+00,copies/cell,ddPCR,B_P03,CLONE_0001,3,early


## 3. Extract clone-level ddPCR feature

ddPCR copy number is measured once per clone early in the process.

We convert it into a single clone-level feature:

- `ddpcr_cn`

This can be useful because copy number may affect productivity and burden.

In [4]:
ddpcr = assay_early[assay_early["assay_type"] == "ddpcr_cn"].copy()
ddpcr = (
    ddpcr.groupby("clone_id", as_index=False)["value"]
    .mean()
    .rename(columns={"value": "ddpcr_cn"})
)

print("ddPCR clones:", len(ddpcr))
display(ddpcr.head())

ddPCR clones: 5000


,clone_id,ddpcr_cn
0,CLONE_0001,4.0
1,CLONE_0002,2.0
2,CLONE_0003,5.0
3,CLONE_0004,4.0
4,CLONE_0005,6.0


## 4. Build passage-level wide table

We reshape assay data into a clone × passage table.

This makes it easier to compute:

- summary statistics
- slopes
- curvature
- qP proxy

In [5]:
wide = assay_early.pivot_table(
    index=["clone_id", "passage_number"],
    columns="assay_type",
    values="value",
    aggfunc="mean"
).reset_index()

wide.columns.name = None
display(wide.head())

,clone_id,passage_number,aggregation,ddpcr_cn,titer,vcd,viability
0,CLONE_0001,3,5.468207,4.0,3.677143,1.134129e+07,90.695877
1,CLONE_0001,4,5.363763,NaN,3.499759,1.032022e+07,93.828926
2,CLONE_0001,5,5.885333,NaN,3.553105,1.178753e+07,95.365890
3,CLONE_0001,6,5.700788,NaN,3.354610,1.103098e+07,91.398229
4,CLONE_0001,7,5.695175,NaN,3.038644,1.149137e+07,95.414562


## 5. Construct qP proxy

We define an early productivity proxy:

- `qP_proxy = titer / VCD`

This is used as a relative productivity feature for clone comparison.

In [6]:
wide["qP_proxy"] = wide["titer"] / wide["vcd"].replace(0, np.nan)
display(wide[["clone_id", "passage_number", "titer", "vcd", "qP_proxy"]].head())

,clone_id,passage_number,titer,vcd,qP_proxy
0,CLONE_0001,3,3.677143,1.134129e+07,3.242260e-07
1,CLONE_0001,4,3.499759,1.032022e+07,3.391166e-07
2,CLONE_0001,5,3.553105,1.178753e+07,3.014292e-07
3,CLONE_0001,6,3.354610,1.103098e+07,3.041082e-07
4,CLONE_0001,7,3.038644,1.149137e+07,2.644285e-07


## 6. Build clone-level summary features

For each early-stage variable, we compute clone-level summaries:

- mean
- std
- min
- max
- coefficient of variation (CV)
- last observed value at P10

These summarize clone behavior within the early screen.

In [7]:
feature_rows = []

for clone_id, sub in wide.groupby("clone_id"):
    sub = sub.sort_values("passage_number").copy()

    row = {"clone_id": clone_id}

    for col in ["titer", "vcd", "viability", "aggregation", "qP_proxy"]:
        vals = sub[col].astype(float)

        row[f"{col}_mean"] = vals.mean()
        row[f"{col}_std"] = vals.std()
        row[f"{col}_min"] = vals.min()
        row[f"{col}_max"] = vals.max()
        row[f"{col}_cv"] = vals.std() / (vals.mean() + 1e-12)

        p10 = sub.loc[sub["passage_number"] == EARLY_END, col]
        row[f"{col}_p10"] = p10.iloc[0] if len(p10) > 0 else np.nan

    feature_rows.append(row)

features = pd.DataFrame(feature_rows)
print("Features shape after summaries:", features.shape)
display(features.head())

Features shape after summaries: (5000, 31)


,clone_id,titer_mean,titer_std,titer_min,titer_max,titer_cv,titer_p10,vcd_mean,vcd_std,vcd_min,...,aggregation_min,aggregation_max,aggregation_cv,aggregation_p10,qP_proxy_mean,qP_proxy_std,qP_proxy_min,qP_proxy_max,qP_proxy_cv,qP_proxy_p10
0,CLONE_0001,3.329594,0.231718,3.038644,3.677143,0.069593,3.211430,1.116369e+07,632529.287976,1.011644e+07,...,5.305646,5.885333,0.037374,5.492092,2.991799e-07,2.768700e-08,2.644285e-07,3.391166e-07,0.092543,3.174466e-07
1,CLONE_0002,0.809159,0.121544,0.646515,1.029448,0.150210,0.857412,1.466332e+07,961363.275997,1.314522e+07,...,6.111981,6.695767,0.033573,6.325135,5.554610e-08,1.009666e-08,4.263966e-08,7.248114e-08,0.181768,5.789301e-08
2,CLONE_0003,5.020162,0.427049,4.426309,5.619000,0.085067,4.512218,9.035878e+06,796851.792897,7.517549e+06,...,1.927987,2.841344,0.125467,2.218711,5.600003e-07,7.722537e-08,4.906684e-07,7.286240e-07,0.137902,5.455803e-07
3,CLONE_0004,1.470888,0.334937,0.949309,1.848296,0.227710,1.124509,1.836343e+07,962793.790256,1.694861e+07,...,0.877338,2.993959,0.362008,2.993959,8.092406e-08,2.151389e-08,4.834680e-08,1.090530e-07,0.265850,6.043763e-08
4,CLONE_0005,3.271652,0.203343,3.045858,3.585816,0.062153,3.045858,1.105298e+07,805278.305154,1.005466e+07,...,2.516000,3.153756,0.084403,3.052792,2.977628e-07,3.224190e-08,2.430777e-07,3.349353e-07,0.108280,2.632503e-07


## 7. Add dynamic early features

We compute dynamic features to capture early trajectory shape:

- full-window slope
- split slopes:
  - P3–6
  - P7–10
- curvature:
  - later slope − earlier slope

These features help distinguish:
- improving clones
- flattening clones
- early false positives

In [8]:
def slope(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan
    return np.polyfit(x[mask], y[mask], 1)[0]

dyn_rows = []

for clone_id, sub in wide.groupby("clone_id"):
    sub = sub.sort_values("passage_number").copy()
    row = {"clone_id": clone_id}

    for col in ["titer", "vcd", "viability", "aggregation", "qP_proxy"]:
        x_all = sub["passage_number"].values
        y_all = sub[col].values

        row[f"{col}_slope"] = slope(x_all, y_all)

        sub_36 = sub[(sub["passage_number"] >= 3) & (sub["passage_number"] <= 6)]
        sub_710 = sub[(sub["passage_number"] >= 7) & (sub["passage_number"] <= 10)]

        s36 = slope(sub_36["passage_number"].values, sub_36[col].values)
        s710 = slope(sub_710["passage_number"].values, sub_710[col].values)

        row[f"{col}_slope_3_6"] = s36
        row[f"{col}_slope_7_10"] = s710
        row[f"{col}_curvature"] = s710 - s36 if np.isfinite(s36) and np.isfinite(s710) else np.nan

    dyn_rows.append(row)

dyn = pd.DataFrame(dyn_rows)
print("Dynamic features shape:", dyn.shape)
display(dyn.head())

Dynamic features shape: (5000, 21)


,clone_id,titer_slope,titer_slope_3_6,titer_slope_7_10,titer_curvature,vcd_slope,vcd_slope_3_6,vcd_slope_7_10,vcd_curvature,viability_slope,...,viability_slope_7_10,viability_curvature,aggregation_slope,aggregation_slope_3_6,aggregation_slope_7_10,aggregation_curvature,qP_proxy_slope,qP_proxy_slope_3_6,qP_proxy_slope_7_10,qP_proxy_curvature
0,CLONE_0001,-0.075758,-0.091425,0.068047,0.159472,-22432.138636,53635.417012,-381856.739617,-435492.156629,0.443062,...,0.054208,-0.310194,-0.005097,0.121931,-0.108922,-0.230854,-6.065050e-09,-9.804111e-09,1.658604e-08,2.639015e-08
1,CLONE_0002,0.017085,-0.014706,0.040328,0.055035,4881.998396,-508708.135152,-323690.893761,185017.241392,0.249612,...,0.819436,0.446730,0.046859,0.099201,0.048396,-0.050806,1.090439e-09,8.269449e-10,3.961753e-09,3.134808e-09
2,CLONE_0003,-0.171314,-0.179189,-0.181397,-0.002208,5475.402927,329656.810312,-485651.321815,-815308.132127,-0.081491,...,-0.432366,-0.323968,-0.000875,0.162507,0.016437,-0.146070,-2.093974e-08,-4.226795e-08,8.275974e-09,5.054393e-08
3,CLONE_0004,-0.107735,-0.056553,-0.146190,-0.089636,240758.145742,761865.641270,209790.066260,-552075.575010,0.546581,...,0.346128,-0.378391,0.071237,-0.272715,0.497228,0.769943,-6.983557e-09,-6.841016e-09,-9.167305e-09,-2.326289e-09
4,CLONE_0005,-0.067631,-0.055974,-0.036062,0.019912,208723.847139,231122.884415,420226.295711,189103.411295,0.378608,...,-0.255089,-1.365594,-0.038663,-0.037569,0.056123,0.093693,-1.144508e-08,-1.218754e-08,-1.332141e-08,-1.133876e-09


## 8. Merge all early-stage features

We merge:

- summary statistics
- dynamic features
- ddPCR feature

to create the final early-stage feature table.

In [9]:
features_v2 = (
    features
    .merge(dyn, on="clone_id", how="left")
    .merge(ddpcr, on="clone_id", how="left")
)

# compatibility alias for downstream notebooks
features_v2["early_mean_qp"] = features_v2["qP_proxy_mean"]

print("Final feature table shape:", features_v2.shape)
display(features_v2.head())

Final feature table shape: (5000, 53)


,clone_id,titer_mean,titer_std,titer_min,titer_max,titer_cv,titer_p10,vcd_mean,vcd_std,vcd_min,...,aggregation_slope,aggregation_slope_3_6,aggregation_slope_7_10,aggregation_curvature,qP_proxy_slope,qP_proxy_slope_3_6,qP_proxy_slope_7_10,qP_proxy_curvature,ddpcr_cn,early_mean_qp
0,CLONE_0001,3.329594,0.231718,3.038644,3.677143,0.069593,3.211430,1.116369e+07,632529.287976,1.011644e+07,...,-0.005097,0.121931,-0.108922,-0.230854,-6.065050e-09,-9.804111e-09,1.658604e-08,2.639015e-08,4.0,2.991799e-07
1,CLONE_0002,0.809159,0.121544,0.646515,1.029448,0.150210,0.857412,1.466332e+07,961363.275997,1.314522e+07,...,0.046859,0.099201,0.048396,-0.050806,1.090439e-09,8.269449e-10,3.961753e-09,3.134808e-09,2.0,5.554610e-08
2,CLONE_0003,5.020162,0.427049,4.426309,5.619000,0.085067,4.512218,9.035878e+06,796851.792897,7.517549e+06,...,-0.000875,0.162507,0.016437,-0.146070,-2.093974e-08,-4.226795e-08,8.275974e-09,5.054393e-08,5.0,5.600003e-07
3,CLONE_0004,1.470888,0.334937,0.949309,1.848296,0.227710,1.124509,1.836343e+07,962793.790256,1.694861e+07,...,0.071237,-0.272715,0.497228,0.769943,-6.983557e-09,-6.841016e-09,-9.167305e-09,-2.326289e-09,4.0,8.092406e-08
4,CLONE_0005,3.271652,0.203343,3.045858,3.585816,0.062153,3.045858,1.105298e+07,805278.305154,1.005466e+07,...,-0.038663,-0.037569,0.056123,0.093693,-1.144508e-08,-1.218754e-08,-1.332141e-08,-1.133876e-09,6.0,2.977628e-07


In [10]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
features_v2.to_csv(OUT_FEATURES, index=False)
print("Saved features:", OUT_FEATURES)

Saved features: ../data/synthetic/processed/cld_features_v2_5000_legacy.csv


## 9. Build late-stage targets from raw DB

We compute late-stage outcomes from the raw database using the late passage window.

Main targets:

- `late_mean_qp`
- `late_mean_aggregation`

Derived targets:

- `qp_drop_pct`
- `stable_label_30pct`

These targets are used only as labels, not as features.

In [11]:
def fetch_late_labels(conn, clone_list, late_start=24, late_end=30):
    """
    Compute late-stage mean qP and aggregation per clone.

    qP proxy is computed passage-wise as:
        qP = titer / vcd
    and then averaged over the late window.
    """
    if len(clone_list) == 0:
        return pd.DataFrame(
            columns=[
                "clone_id",
                "late_mean_titer",
                "late_mean_vcd",
                "late_mean_qp",
                "late_mean_aggregation",
            ]
        )

    placeholders = ",".join(["?"] * len(clone_list))

    query = f"""
    WITH late_assays AS (
        SELECT
            p.clone_id,
            p.passage_number,
            AVG(CASE WHEN ar.assay_type = 'titer' THEN ar.value END) AS titer_value,
            AVG(CASE WHEN ar.assay_type = 'vcd' THEN ar.value END) AS vcd_value,
            AVG(CASE WHEN ar.assay_type = 'aggregation' THEN ar.value END) AS aggregation_value
        FROM assay_result ar
        JOIN passage p
          ON p.passage_id = ar.passage_id
        WHERE p.passage_number BETWEEN ? AND ?
          AND p.clone_id IN ({placeholders})
        GROUP BY p.clone_id, p.passage_number
    )
    SELECT
        clone_id,
        AVG(titer_value) AS late_mean_titer,
        AVG(vcd_value) AS late_mean_vcd,
        AVG(titer_value / NULLIF(vcd_value, 0)) AS late_mean_qp,
        AVG(aggregation_value) AS late_mean_aggregation
    FROM late_assays
    GROUP BY clone_id
    """

    params = [late_start, late_end] + list(clone_list)
    return pd.read_sql_query(query, conn, params=params)

In [12]:
conn = sqlite3.connect(DB_PATH)

late_labels = fetch_late_labels(
    conn,
    features_v2["clone_id"].tolist(),
    late_start=LATE_START,
    late_end=LATE_END
)

conn.close()
display(late_labels.head())

,clone_id,late_mean_titer,late_mean_vcd,late_mean_qp,late_mean_aggregation
0,CLONE_0001,2.081741,1.451080e+07,1.437208e-07,8.173122
1,CLONE_0002,0.585364,1.642887e+07,3.568823e-08,7.854971
2,CLONE_0003,2.769028,1.297029e+07,2.147891e-07,1.142243
3,CLONE_0004,0.325381,1.732515e+07,1.895158e-08,2.126247
4,CLONE_0005,2.196352,1.409948e+07,1.568152e-07,4.649927


In [13]:
dataset = features_v2.merge(
    late_labels,
    on="clone_id",
    how="left"
)

dataset["early_mean_qp"] = dataset["qP_proxy_mean"]

dataset["qp_drop_pct"] = (
    (dataset["early_mean_qp"] - dataset["late_mean_qp"])
    / (dataset["early_mean_qp"] + 1e-12)
).clip(lower=0.0, upper=1.5)

dataset["stable_label_30pct"] = (
    dataset["qp_drop_pct"] <= STABILITY_DROP_THRESHOLD
).astype(int)

display(dataset.head())

,clone_id,titer_mean,titer_std,titer_min,titer_max,titer_cv,titer_p10,vcd_mean,vcd_std,vcd_min,...,qP_proxy_slope_7_10,qP_proxy_curvature,ddpcr_cn,early_mean_qp,late_mean_titer,late_mean_vcd,late_mean_qp,late_mean_aggregation,qp_drop_pct,stable_label_30pct
0,CLONE_0001,3.329594,0.231718,3.038644,3.677143,0.069593,3.211430,1.116369e+07,632529.287976,1.011644e+07,...,1.658604e-08,2.639015e-08,4.0,2.991799e-07,2.081741,1.451080e+07,1.437208e-07,8.173122,0.519616,0
1,CLONE_0002,0.809159,0.121544,0.646515,1.029448,0.150210,0.857412,1.466332e+07,961363.275997,1.314522e+07,...,3.961753e-09,3.134808e-09,2.0,5.554610e-08,0.585364,1.642887e+07,3.568823e-08,7.854971,0.357496,0
2,CLONE_0003,5.020162,0.427049,4.426309,5.619000,0.085067,4.512218,9.035878e+06,796851.792897,7.517549e+06,...,8.275974e-09,5.054393e-08,5.0,5.600003e-07,2.769028,1.297029e+07,2.147891e-07,1.142243,0.616447,0
3,CLONE_0004,1.470888,0.334937,0.949309,1.848296,0.227710,1.124509,1.836343e+07,962793.790256,1.694861e+07,...,-9.167305e-09,-2.326289e-09,4.0,8.092406e-08,0.325381,1.732515e+07,1.895158e-08,2.126247,0.765801,0
4,CLONE_0005,3.271652,0.203343,3.045858,3.585816,0.062153,3.045858,1.105298e+07,805278.305154,1.005466e+07,...,-1.332141e-08,-1.133876e-09,6.0,2.977628e-07,2.196352,1.409948e+07,1.568152e-07,4.649927,0.473354,0


In [14]:
dataset[
    [
        "early_mean_qp",
        "late_mean_qp",
        "qp_drop_pct",
        "stable_label_30pct",
        "late_mean_aggregation",
    ]
].isna().mean()

early_mean_qp            0.0
late_mean_qp             0.0
qp_drop_pct              0.0
stable_label_30pct       0.0
late_mean_aggregation    0.0
dtype: float64

In [15]:
dataset[
    [
        "early_mean_qp",
        "late_mean_qp",
        "qp_drop_pct",
        "stable_label_30pct",
        "late_mean_aggregation",
    ]
].describe()

,early_mean_qp,late_mean_qp,qp_drop_pct,stable_label_30pct,late_mean_aggregation
count,5.000000e+03,5.000000e+03,5000.000000,5000.000000,5000.000000
mean,3.319901e-07,2.260904e-07,0.450371,0.167600,6.178710
std,1.520843e-06,5.048462e-06,0.162605,0.373548,3.266330
min,8.969234e-09,1.466020e-09,0.000000,0.000000,0.006032
25%,8.187386e-08,4.290678e-08,0.339551,0.000000,3.678536
50%,1.462656e-07,7.896058e-08,0.433724,0.000000,5.911444
75%,2.711440e-07,1.465324e-07,0.546566,0.000000,8.344825
max,6.959865e-05,3.556629e-04,0.983312,1.000000,20.368550


## 10. Enforce canonical qP-based schema

To keep downstream notebooks stable, we remove obsolete titer-based columns if they exist.

- **Canonical schema** means the standard column set expected by later notebooks.
- This prevents mismatch between old and new naming systems.

In [16]:
legacy_cols = ["late_mean_titer", "late_mean_vcd", "productivity_drop_pct"]
to_drop = [c for c in legacy_cols if c in dataset.columns]

if to_drop:
    print("Removing legacy columns before save:", to_drop)
    dataset = dataset.drop(columns=to_drop)

print("\nSchema check after cleanup:")
for c in [
    "early_mean_qp",
    "late_mean_qp",
    "qp_drop_pct",
    "stable_label_30pct",
    "late_mean_aggregation",
    "late_mean_titer",
    "late_mean_vcd",
    "productivity_drop_pct",
]:
    print(f"{c}: {c in dataset.columns}")

Removing legacy columns before save: ['late_mean_titer', 'late_mean_vcd']

Schema check after cleanup:
early_mean_qp: True
late_mean_qp: True
qp_drop_pct: True
stable_label_30pct: True
late_mean_aggregation: True
late_mean_titer: False
late_mean_vcd: False
productivity_drop_pct: False


In [17]:
dataset.to_csv(OUT_DATASET, index=False)
print("Saved dataset:", OUT_DATASET)
print("Rows:", dataset.shape[0], "Columns:", dataset.shape[1])

Saved dataset: ../data/synthetic/processed/cld_features_with_labels_qp_targets_v2_24_30_5000_legacy.csv
Rows: 5000 Columns: 57


## Output of Notebook 02

This notebook produces two outputs:

### 1. Early-stage feature table
Clone-level features from early passages only.

### 2. Canonical modeling dataset
A leakage-safe dataset containing:

- early-stage features
- late-stage qP target
- stability target
- late-stage aggregation target

---

## Downstream usage

This output is used by:

- **Notebook 03b** — prediction engine
- **Notebook 04b** — decision engine

---

## Pipeline meaning

- Notebook 01 = understand the synthetic data
- Notebook 02 = define X and y
- Notebook 03 = predict late outcomes
- Notebook 04 = make clone selection decisions

In [18]:
print(dataset.shape)

print(
    dataset[
        ["early_mean_qp", "late_mean_qp", "qp_drop_pct", "stable_label_30pct", "late_mean_aggregation"]
    ].head()
)

print(
    dataset[
        ["qp_drop_pct", "stable_label_30pct", "late_mean_qp", "late_mean_aggregation"]
    ].describe()
)

(5000, 57)
   early_mean_qp  late_mean_qp  qp_drop_pct  stable_label_30pct  \
0   2.991799e-07  1.437208e-07     0.519616                   0   
1   5.554610e-08  3.568823e-08     0.357496                   0   
2   5.600003e-07  2.147891e-07     0.616447                   0   
3   8.092406e-08  1.895158e-08     0.765801                   0   
4   2.977628e-07  1.568152e-07     0.473354                   0   

   late_mean_aggregation  
0               8.173122  
1               7.854971  
2               1.142243  
3               2.126247  
4               4.649927  
       qp_drop_pct  stable_label_30pct  late_mean_qp  late_mean_aggregation
count  5000.000000         5000.000000  5.000000e+03            5000.000000
mean      0.450371            0.167600  2.260904e-07               6.178710
std       0.162605            0.373548  5.048462e-06               3.266330
min       0.000000            0.000000  1.466020e-09               0.006032
25%       0.339551            0.000000  4.29